# GeoFire-X — B1 CNN, données réelles, T=3
Notebook autonome : code + métadonnées intégrés, aucun GeoTIFF intégré. Attacher les datasets déjà publiés `nnnnbhss/geofirex-wsts-fold0-tv-001-v1`, `nnnnbhss/geofirex-wsts-fold0-tv-002-v1` et `nnnnbhss/geofire-x-wildfirespreadts-sample` (validation 2020 historique). Aucun re-upload. Les 119 événements train des deux lots sont un inventaire historique certifié, vérifié à nouveau au montage. La validation disponible dans le sample reste limitée à deux événements : aucune revendication sur les 201. Alternative : attacher le corpus compact après autorisation de transfert.

Le profil effectue de vraies mises à jour puis évalue. Les époques complètes sont dans une cellule séparée, à exécuter après accord sur le budget estimé. Aucun job ne se lance à l’ouverture. Garder les sorties Kaggle avant arrêt de session. Aucun test 2021.

In [ ]:
import base64, hashlib, io, zipfile, sys, json, time, subprocess
from pathlib import Path
payload=base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest()=='07d70e66abee11791cc79f2af2cb1263a5f7db9d21ee9af001f20af2d8faebce'
workspace=Path("/kaggle/working/geofirex-mission-code")
workspace.mkdir(exist_ok=False)
with zipfile.ZipFile(io.BytesIO(payload)) as z: z.extractall(workspace)
sys.path.insert(0,str(workspace/"src"))
prepared=workspace/"metadata"
print("Code+metadata verified:",len(payload),"bytes")

In [ ]:
import torch, numpy as np, rasterio, pyarrow
assert torch.cuda.is_available(), "Activer le GPU Kaggle; ne pas remplacer torch"
print(torch.__version__,torch.version.cuda,torch.cuda.get_device_name(0))
print(subprocess.run(["nvidia-smi"],capture_output=True,text=True).stdout)
print("Dataset mounts:",[str(p) for p in Path("/kaggle/input").iterdir()])
# Le quota restant doit être lu dans l’interface Kaggle : il n’est pas déduit du GPU alloué.
# Aucun téléchargement, transfert ou installation automatique.

In [ ]:
from geofirex.mission_streaming import CompactTiles,save
from geofirex.mission_training import attached_datasets,train
compact_roots=[p.parent for p in Path("/kaggle/input").rglob("samples.json") if (p.parent/"normalization.json").is_file()]
assert len(compact_roots)<=1, "Choisir explicitement un corpus compact si plusieurs sont attachés"
if compact_roots:
    corpus=compact_roots[0]
    train_ds,val_ds=CompactTiles(corpus,"train"),CompactTiles(corpus,"val")
    provenance={"source":"compact", "root":str(corpus)}
else:
    roots=sorted({p.parent for p in Path("/kaggle/input").rglob("2018") if p.is_dir()} | {p.parent for p in Path("/kaggle/input").rglob("2020") if p.is_dir()})
    assert roots,"Aucun montage raster identifié"
    train_ds,val_ds,provenance=attached_datasets(prepared,roots)
print("train tiles",len(train_ds),"val tiles",len(val_ds))
print("train events",len({s["event_id"] for s in train_ds.samples}),"val events",len({s["event_id"] for s in val_ds.samples}))
from torch.utils.data import DataLoader
x,y,v=next(iter(DataLoader(train_ds,batch_size=4,num_workers=0)))
assert x.shape[1:]==(3,8,64,64) and torch.isfinite(x).all()
print(x.shape, y.shape, v.shape)

In [ ]:
stamp=time.strftime("%Y%m%d_%H%M%S")
profile=train(train_ds,val_ds,Path("/kaggle/working")/("B1-profile-"+stamp),device="cuda",epochs=1,profile_steps=25,wall_seconds=180,provenance=provenance)
print(json.dumps(profile,indent=2))
# Includes full validation time, so conservative for a training-only estimate.
estimated_epoch_seconds=profile["elapsed_seconds"]*len(train_ds)/max(1,profile["train_samples_seen"])
print("Conservative seconds/epoch",estimated_epoch_seconds)

Exécuter la cellule suivante uniquement après autorisation d’une campagne GPU prolongée et vérification du quota courant. Elle démarre un nouveau B1 avec la même seed : les mises à jour du profil ne sont pas mélangées aux résultats. Chaque époque complète visite toutes les tuiles train une fois ; la validation est exhaustive et non rééquilibrée. Le délai maximum coupe le run avec un statut explicite et conserve le dernier checkpoint produit. Les checkpoints incluent modèle, optimiseur, RNG, normalisation liée et comptes réellement vus.

In [ ]:
RUN_FULL_EPOCHS=False  # Passer à True après autorisation du budget GPU
if RUN_FULL_EPOCHS:
    result=train(train_ds,val_ds,Path("/kaggle/working")/("B1-full-"+time.strftime("%Y%m%d_%H%M%S")),device="cuda",epochs=3,wall_seconds=1800,provenance=provenance)
    print(json.dumps(result,indent=2))
    from IPython.display import Image,display
    display(Image(filename=str(sorted(Path("/kaggle/working").glob("B1-full-*/loss.png"))[-1])))